# Livestock Feed Supply-Chain Stress Testing (Prototype)

This notebook compares a **synthetic baseline** with a 20% ingredient-price shock, a supplier-delay scenario, and a combined scenario. It reuses the CSV outputs from the previous optimizer notebook.

**Important scope limits:** the ingredient prices, inventory, daily demand, order quantities, and lead times in the optimizer outputs are illustrative. The mandi CSV is a one-day snapshot and is not automatically merged because ingredient equivalence/units need review. This is an algorithm demonstration, not a validated cattle ration or real farm forecast.

## What this notebook measures
- 30-day feed consumption and inventory levels per ingredient
- Stockout days and unmet demand under each scenario
- Estimated 30-day feed cost under price changes
- A scenario comparison CSV and ZIP export

In [ ]:
from pathlib import Path
import zipfile, shutil, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/livestock_stress_test')
WORK.mkdir(parents=True, exist_ok=True)

print('CSV files visible to this notebook:')
for p in INPUT_ROOT.rglob('*.csv'):
    print(p)


## 1. Load the previous optimizer outputs

In [ ]:
def find_csv(filename):
    matches = list(INPUT_ROOT.rglob(filename))
    if not matches:
        # Also allow users to place prior results in /kaggle/working
        matches = list(Path('/kaggle/working').rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Attach the dataset containing your optimizer result CSVs via Add Input."
        )
    return matches[0]

ration = pd.read_csv(find_csv('illustrative_ration_solution_synthetic.csv'))
reorder = pd.read_csv(find_csv('inventory_reorder_points_synthetic.csv'))

print('Ration shape:', ration.shape)
print('Reorder table shape:', reorder.shape)
display(ration)
display(reorder)


## 2. Configure scenarios

In [ ]:
HORIZON_DAYS = 30
PRICE_SHOCK = 0.20          # +20% ingredient price
EXTRA_DELIVERY_DELAY = 3   # extra days added to every order's lead time in delay scenarios

# Use the daily ration from the previous optimizer as the daily demand per ingredient.
# Initial stock, lead time and order quantity come from the earlier synthetic inventory table.
base = ration.merge(reorder[['ingredient','stock_kg','lead_time_days','order_qty_kg','reorder_point_kg']],
                    on='ingredient', how='inner')
if len(base) != len(ration):
    raise ValueError('Some ration ingredients did not match inventory rows. Check ingredient names.')
base['daily_demand_kg'] = base['optimized_kg_day'].astype(float)
base['base_price_inr_kg'] = base['cost_inr_per_kg'].astype(float)
base['initial_stock_kg'] = base['stock_kg'].astype(float)
base['lead_time_days'] = base['lead_time_days'].astype(int)
base['order_qty_kg'] = base['order_qty_kg'].astype(float)
base['reorder_point_kg'] = base['reorder_point_kg'].astype(float)

print('Scenarios: baseline, price_shock, supplier_delay, combined')
display(base[['ingredient','daily_demand_kg','initial_stock_kg','base_price_inr_kg','lead_time_days','order_qty_kg','reorder_point_kg']])


## 3. Run a transparent inventory simulation

In [ ]:
def simulate_scenario(scenario_name, price_multiplier=1.0, extra_delay_days=0):
    daily_rows = []
    summary_rows = []
    for _, r in base.iterrows():
        stock = float(r['initial_stock_kg'])
        demand = float(r['daily_demand_kg'])
        lead = int(r['lead_time_days']) + int(extra_delay_days)
        order_qty = float(r['order_qty_kg'])
        reorder_point = float(r['reorder_point_kg'])
        # Arrival queue: (arrival_day, quantity). Orders placed on day d arrive on d + lead.
        arrivals = []
        stockout_days = 0
        total_unmet = 0.0
        orders_placed = 0
        total_arrived = 0.0
        min_stock = stock

        for day in range(1, HORIZON_DAYS + 1):
            arriving_today = sum(q for arrival_day, q in arrivals if arrival_day == day)
            arrivals = [(a, q) for a, q in arrivals if a != day]
            stock += arriving_today
            total_arrived += arriving_today

            available_before_demand = stock
            supplied = min(stock, demand)
            unmet = max(0.0, demand - supplied)
            stock -= supplied
            if unmet > 1e-9:
                stockout_days += 1
                total_unmet += unmet

            # Inventory position = on-hand stock + outstanding orders.
            outstanding = sum(q for _, q in arrivals)
            inventory_position = stock + outstanding
            placed = False
            if inventory_position <= reorder_point and order_qty > 0:
                arrival_day = day + max(1, lead)
                arrivals.append((arrival_day, order_qty))
                orders_placed += 1
                placed = True

            min_stock = min(min_stock, stock)
            daily_rows.append({
                'scenario': scenario_name, 'day': day, 'ingredient': r['ingredient'],
                'starting_available_including_arrivals_kg': available_before_demand,
                'daily_demand_kg': demand, 'supplied_kg': supplied, 'unmet_demand_kg': unmet,
                'ending_stock_kg': stock, 'arrivals_today_kg': arriving_today,
                'order_placed': placed, 'outstanding_order_count': len(arrivals),
                'lead_time_days_used': lead,
            })

        unit_price = float(r['base_price_inr_kg']) * price_multiplier
        estimated_30d_feed_cost = demand * HORIZON_DAYS * unit_price
        summary_rows.append({
            'scenario': scenario_name, 'ingredient': r['ingredient'],
            'daily_demand_kg': demand, 'unit_price_inr_kg': unit_price,
            'estimated_30d_consumption_cost_inr': estimated_30d_feed_cost,
            'stockout_days': stockout_days, 'total_unmet_demand_kg': total_unmet,
            'orders_placed': orders_placed, 'minimum_ending_stock_kg': min_stock,
            'extra_delay_days': int(extra_delay_days), 'price_multiplier': price_multiplier,
        })
    return pd.DataFrame(daily_rows), pd.DataFrame(summary_rows)

scenario_specs = [
    ('baseline', 1.0, 0),
    ('price_shock', 1.0 + PRICE_SHOCK, 0),
    ('supplier_delay', 1.0, EXTRA_DELIVERY_DELAY),
    ('combined', 1.0 + PRICE_SHOCK, EXTRA_DELIVERY_DELAY),
]
all_daily, all_summary = [], []
for name, mult, delay in scenario_specs:
    daily, summary = simulate_scenario(name, mult, delay)
    all_daily.append(daily)
    all_summary.append(summary)

daily_results = pd.concat(all_daily, ignore_index=True)
ingredient_summary = pd.concat(all_summary, ignore_index=True)

daily_results.to_csv(WORK / 'scenario_daily_inventory.csv', index=False)
ingredient_summary.to_csv(WORK / 'scenario_ingredient_summary.csv', index=False)
print('Simulation completed.')
display(ingredient_summary.head(12))


## 4. Compare scenarios

In [ ]:
scenario_comparison = ingredient_summary.groupby('scenario', as_index=False).agg(
    estimated_30d_feed_cost_inr=('estimated_30d_consumption_cost_inr','sum'),
    total_stockout_days_across_ingredients=('stockout_days','sum'),
    total_unmet_demand_kg=('total_unmet_demand_kg','sum'),
    total_orders_placed=('orders_placed','sum'),
)
base_cost = float(scenario_comparison.loc[scenario_comparison.scenario == 'baseline', 'estimated_30d_feed_cost_inr'].iloc[0])
scenario_comparison['cost_change_vs_baseline_inr'] = scenario_comparison['estimated_30d_feed_cost_inr'] - base_cost
scenario_comparison['cost_change_vs_baseline_pct'] = np.where(
    base_cost > 0,
    100 * scenario_comparison['cost_change_vs_baseline_inr'] / base_cost,
    np.nan
)
scenario_comparison.to_csv(WORK / 'scenario_comparison.csv', index=False)
display(scenario_comparison)


In [ ]:
# Visual comparison: cost and supply disruption metrics
fig, ax = plt.subplots()
ax.bar(scenario_comparison['scenario'], scenario_comparison['estimated_30d_feed_cost_inr'])
ax.set_title('Estimated 30-day feed consumption cost by scenario')
ax.set_ylabel('INR (synthetic prices)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()
ax.bar(scenario_comparison['scenario'], scenario_comparison['total_unmet_demand_kg'])
ax.set_title('Unmet feed demand by scenario')
ax.set_ylabel('kg across ingredients over 30 days')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()


## 5. Save the outputs as a ZIP

In [ ]:
# Include all generated CSVs and a short methodology note.
(WORK / 'README_scenario_testing.txt').write_text(
    'Livestock feed supply-chain scenario testing prototype\n'
    'Scenarios: baseline, +20% synthetic ingredient price shock, +3 day supplier delay, combined.\n'
    'Inventory and prices are synthetic/illustrative and are not real farm forecasts or validated cattle nutrition advice.\n'
    'The cost metric estimates 30-day consumed feed at the scenario unit prices; it is not a procurement cash-flow total.\n'
    'Stockout metrics depend on the stated starting stocks, demands, order quantities, reorder points and delivery delays.\n'
    'Validate ingredient mappings, units, local prices and nutritional constraints with appropriate sources and experts before real-world use.\n'
)
ZIP_PATH = '/kaggle/working/livestock_stress_test_results.zip'
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
shutil.make_archive('/kaggle/working/livestock_stress_test_results', 'zip', root_dir=WORK)
print('Created:', ZIP_PATH)
print('Files included:')
for p in sorted(WORK.iterdir()):
    print(p.name, '-', p.stat().st_size, 'bytes')


In [ ]:
from IPython.display import FileLink, display

display(FileLink('/kaggle/working/livestock_stress_test_results.zip'))


## Interpretation checklist
- A price shock should raise the estimated feed-consumption cost while leaving inventory mechanics unchanged.
- A delivery delay may increase stockout risk, depending on starting stock and reorder settings.
- If stockouts remain zero, report that result as specific to these simulated assumptions—not proof of real-world resilience.
- The current model does **not** yet re-optimize the ration under each scenario or choose between real suppliers. That is the next optimization extension after scenario outputs are verified.